# Fuel Shock Stress Test: analysis notebook

Layer 3 of the project: oil price behavior, the company screen, and the survival simulation.

Sections:
1. Oil and diesel price history (public data, fetched by `src/fetch_public_data.py`)
2. Shock episode catalog (2008, 2011, 2022) and crude-to-diesel pass-through
3. Company screen (CapitalIQ export, local only, never committed)
4. Survival analysis: applying the shock engine to the empirical margin distribution
5. Monte Carlo: mean-reverting oil paths and probability of margin breach

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

brent = pd.read_csv('../data/public/brent_usd_bbl_daily.csv', parse_dates=['date'])
diesel = pd.read_csv('../data/public/us_diesel_usd_gal_weekly.csv', parse_dates=['date'])
print(f'Brent: {len(brent)} rows to {brent.date.max().date()}')
print(f'US diesel: {len(diesel)} rows to {diesel.date.max().date()}')

## 1. Price history and realized volatility

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(brent.date, brent.value, lw=0.7)
axes[0].set_title('Brent spot, USD per barrel')
ret = np.log(brent.value).diff()
vol30 = ret.rolling(30).std() * np.sqrt(252)
axes[1].plot(brent.date, vol30, lw=0.7, color='darkred')
axes[1].set_title('30-day realized volatility, annualized')
plt.tight_layout()
plt.show()

## 2. Shock episodes and crude-to-diesel pass-through

Merge Brent (resampled weekly) with US retail diesel and estimate the pass-through beta:
how much of a percentage move in crude reaches the pump, and with what lag.
Repeat per country in week 2 with Eurostat, PPAC and ANP data to quantify the tax wedge.

In [ ]:
bw = brent.set_index('date').value.resample('W-MON').mean()
dw = diesel.set_index('date').value
merged = pd.concat([bw.rename('brent'), dw.rename('diesel')], axis=1).dropna()
chg = merged.pct_change(13).dropna()   # quarterly moves
beta = np.polyfit(chg.brent, chg.diesel, 1)[0]
print(f'US crude-to-pump beta (13-week moves, {chg.index.min().date()} on): {beta:.2f}')

## 3. Company screen (CapitalIQ export)

Drop the CIQ export at `../data/licensed/ciq_screen.xlsx` (gitignored).
Expected fields per company: name, ticker, country, market cap, revenue, EBITDA margin,
EBIT margin (FY2015 to FY2025), cash and short-term investments, total debt,
interest expense, employees.

In [ ]:
# ciq = pd.read_excel('../data/licensed/ciq_screen.xlsx')
# ciq.head()

## 4. Survival analysis

Apply the shock engine to each company: shocked margin = baseline margin minus
fuel share x pump shock x (1 minus effective pass-through). Companies without fuel
cost disclosure get the country-mode fuel share from the Excel model. Output:
share of companies breaching zero margin and share breaching cash within the duration.

## 5. Monte Carlo (mean-reverting oil)

Ornstein-Uhlenbeck on log Brent calibrated on 2010 to 2025: oil mean-reverts, so GBM
would overstate long-horizon drift. Simulate 10,000 12-month paths, run each through
the shock engine, and report the probability of margin breach by country and fleet size.

In [ ]:
# Calibration sketch (week 3):
# x = np.log(brent.set_index('date').value['2010':])
# dx, lag = x.diff().dropna(), x.shift(1).dropna()
# theta, mu, sigma from OLS of dx on lag (AR(1) to OU mapping)
pass